![A soccer pitch for an international match.](soccer-pitch.jpg)

You're working as a sports journalist at a major online sports media company, specializing in soccer analysis and reporting. You've been watching both men's and women's international soccer matches for a number of years, and your gut instinct tells you that more goals are scored in women's international football matches than men's. This would make an interesting investigative article that your subscribers are bound to love, but you'll need to perform a valid statistical hypothesis test to be sure!

While scoping this project, you acknowledge that the sport has changed a lot over the years, and performances likely vary a lot depending on the tournament, so you decide to limit the data used in the analysis to only official `FIFA World Cup` matches (not including qualifiers) since `2002-01-01`.

You create two datasets containing the results of every official men's and women's international football match since the 19th century, which you scraped from a reliable online source. This data is stored in two CSV files: `women_results.csv` and `men_results.csv`.

The question you are trying to determine the answer to is:

> Are more goals scored in women's international soccer matches than men's?

You assume a **10% significance level**, and use the following null and alternative hypotheses:

$H_0$ : The mean number of goals scored in women's international soccer matches is the same as men's.

$H_A$ : The mean number of goals scored in women's international soccer matches is greater than men's.

In [14]:
# Start your code here!
import pandas as pd
import pingouin as pg
import matplotlib.pyplot as plt

men_results = pd.read_csv('men_results.csv')
women_results = pd.read_csv('women_results.csv')

#men
men_results['date'] = pd.to_datetime(men_results['date'])

men_fifa_world_cup_matches = men_results[
    (men_results['tournament'].isin(["FIFA World Cup"])) &
    (men_results['date'] > '2002-01-01')
]

#women
women_results['date'] = pd.to_datetime(women_results['date'])

women_fifa_world_cup_matches = women_results[
    (women_results['tournament'].isin(["FIFA World Cup"])) &
    (women_results['date'] > '2002-01-01')
]

#adding new columns to respective data sets.
men_fifa_world_cup_matches['gender'] = 'Men'
women_fifa_world_cup_matches['gender'] = 'Women'

men_fifa_world_cup_matches['goals_scored'] = men_fifa_world_cup_matches['home_score'] + men_fifa_world_cup_matches['away_score']
women_fifa_world_cup_matches['goals_scored'] = women_fifa_world_cup_matches['home_score'] + women_fifa_world_cup_matches['away_score']

# #check if normally distributed to determine hypothesis test.
# men_fifa_world_cup_matches['goals_scored'].hist()
# plt.show()
# plt.clf()

# women_fifa_world_cup_matches['goals_scored'].hist()
# plt.show()
# plt.clf()

#concat both data sets
combined = pd.concat([men_fifa_world_cup_matches, women_fifa_world_cup_matches], axis=0, ignore_index=True)

#transform dataset into wide-format, appropriate for pingouin
combined_wide =  combined.pivot(columns="gender", values="goals_scored")

#Perform right tailed test as per question: Are more goals scored in women's international soccer matches than men's?
results_pg = pg.mwu(x=combined_wide['Women'], y=combined_wide['Men'], alternative="greater")

print(results_pg)

#Extract p-val as a float
p_val = results_pg["p-val"].values[0]

#Determine hypothesis test result using significance level
if p_val <= 0.1:
    result = "reject"
else:
    result = "fail to reject"

result_dict = {"p_val":p_val, "result":result}

       U-val alternative     p-val       RBC      CLES
MWU  43273.0     greater  0.005107 -0.126901  0.563451
